# Multimodal Fine Tuned Representations on LLaVA: Layerwise vs Headwise Analysis

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown
from IPython.display import clear_output
from scripts.analyze_heads_representations import plot_overlap_heatmap

## Functions

In [ ]:
def binary_op(A_bin, B_bin, operation):
    if operation == "xor":
        return np.logical_xor(A_bin, B_bin).astype(int)
    elif operation == "or":
        return np.logical_or(A_bin, B_bin).astype(int)
    elif operation == "and":
        return np.logical_and(A_bin, B_bin).astype(int)
    else:
        raise ValueError(f"Unknown operation '{operation}'")


def visualize_matrix(A, B, threshold, operation):
    clear_output(wait=True)  # This clears previous output in the cell
    A_bin = A > threshold
    B_bin = B > threshold
    result = binary_op(A_bin, B_bin, operation)

    plt.figure(figsize=(5, 5))
    plt.imshow(result, cmap="gray", vmin=0, vmax=1)
    plt.title(
        rf"${{{operation.upper()}}}: M^{{\text{{text}}}}_{{\geq \tau}} \oplus M^{{\text{{images}}}}_{{\geq \tau}},\ \tau = {threshold:.2f}$"
    )
    plt.axis("off")
    plt.show()

# Dropdown for operation and slider for threshold
def binary_heads_change(A, B):
    interact(
        lambda t, op: visualize_matrix(A, B, t, op),
        t=FloatSlider(min=0, max=1, step=0.01, value=0.5, description="Threshold"),
        op=Dropdown(options=["xor", "or", "and"], value="xor", description="Operation"),
    )
    
def visualize_matrix_subplot(ax, A, B, threshold, operation):
    A_bin = A >= threshold
    B_bin = B <= threshold
    result = binary_op(A_bin, B_bin, operation)

    ax.imshow(result, cmap="gray", vmin=0, vmax=1)
    ax.set_title(rf"$\tau = {threshold:.2f}$", fontsize=10)
    ax.axis("off")

# Main plotting code
def plot_all_thresholds(A, B, thresholds, operation, rows=4, cols=3):
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = axes.flatten()  # Flatten in case it's 2D

    for i, threshold in enumerate(thresholds):
        if i < len(axes):
            visualize_matrix_subplot(axes[i], A, B, threshold, operation)

    # Hide any unused axes (if thresholds < rows * cols)
    for j in range(len(thresholds), len(axes)):
        axes[j].axis("off")

    fig.suptitle(
        rf"${operation.upper()}: M^{{\text{{text}}}}_{{\geq \tau}} \oplus M^{{\text{{images}}}}_{{\geq \tau}}$", fontsize=16
    )

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Leave space for suptitle
    plt.show()

def remove_prefixes_str_from_rows_and_columns(df):
    """
    Removes prefixes 'L' from index and 'H' from columns in the DataFrame.
    """
    df.index = df.index.str.replace(r"^L?", "", regex=True)
    df.columns = df.columns.str.replace(r"^H?", "", regex=True)
    

def plot_box_plot_layerwise_stats_across_heads(
    df,
    dataset_name="Dataset",
):
    """
    Plots a box plot of layerwise statistics across heads.
    Parameters:
    - df: pandas DataFrame with index like L0, L1, ..., L31 and columns like 'mean', 'std', 'min', etc.
    """

    # Transpose the DataFrame so that rows become columns for plotting
    plt.boxplot(df.T, vert=False)
    plt.yticks(
        ticks=range(1, len(df) + 1),
        labels=df.index,
    )
    plt.xlim(0, 1)
    plt.xlabel("Neighborhood Overlap (LLaVa 1.5 7b vs Vicuna 1.5 7B)")
    plt.ylabel("Layer Index")
    plt.title(f"Layerwise Statistics of Heads Representations ({dataset_name})")
    plt.show()


def compute_layers_stats_across_heads(df):
    return df.apply(
        lambda row: pd.Series(
            {
                "mean": row.mean(),
                "std": row.std(),
                "min": row.min(),
                "max": row.max(),
                "median": row.median(),
            }
        ),
        axis=1,
    )


def plot_layerwise_stats_across_heads(
    df,
    figsize=(12, 6),
    dataset_name="Dataset",
):
    """
    Plots selected row-wise statistics against layer index.

    Parameters:
    - df: pandas DataFrame with index like L0, L1, ..., L31 and columns like 'mean', 'std', 'min', etc.
    - figsize: tuple, size of the figure
    - title: plot title
    """

    x = df.index.to_numpy()
    mean = df["mean"].to_numpy()
    std = df["std"].to_numpy()
    upper = mean + std
    lower = mean - std

    stats_to_plot = df.columns.difference(["mean", "std"])

    # Plot
    plt.figure(figsize=figsize)
    for stat in stats_to_plot:
        plt.plot(df[stat], marker="o", label=stat)

    # Plot mean and std as band
    plt.plot(x, mean, marker="o", color="purple", label="Mean", linewidth=2)
    plt.fill_between(x, lower, upper, color="blue", alpha=0.2, label="±1 Std Dev")

    plt.xlabel("Layer Index")
    plt.ylabel("Neighborhood Overlap Moments Value")
    plt.ylim(0, 1)
    plt.title(dataset_name)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def plot_overlap_layerwise_vs_heads_mean(
    layerwise_overlap_df,
    heads_mean_overlap_np,
    dataset_name="Dataset",
    figsize=(12, 6),
):
    x = layerwise_overlap_df.index.to_numpy()

    plt.figure(figsize=figsize)
    plt.plot(
        x,
        layerwise_overlap_df["Neighborhood Overlap"],
        marker="o",
        label="Layerwise",
    )
    plt.plot(
        x,
        heads_mean_overlap_np,
        marker="o",
        label="Heads Mean",
    )
    plt.xlabel("Layer Index")
    plt.ylabel("Neighborhood Overlap")
    plt.ylim(0, 1)
    plt.title(dataset_name)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def plot_overlap_heads_moments_vs_layerwise(
    overlap_heads_moments_df,
    layerwise_overlap_df,
    figsize=(12, 6),
    dataset_name="Dataset",
):
    """
    Plots selected row-wise statistics against layer index.

    Parameters:
    - overlap_heads_moments_df: pandas DataFrame with index like L0, L1, ..., L31 and columns like 'mean', 'std', 'min', etc.
    - layerwise_overlap_df: pandas DataFrame with layerwise overlap statistics
    - figsize: tuple, size of the figure
    - title: plot title
    """

    x = overlap_heads_moments_df.index.to_numpy()
    mean = overlap_heads_moments_df["mean"].to_numpy()
    std = overlap_heads_moments_df["std"].to_numpy()
    upper = mean + std
    lower = mean - std

    stats_to_plot = overlap_heads_moments_df.columns.difference(["mean", "std"])

    # Plot
    plt.figure(figsize=figsize)
    for stat in stats_to_plot:
        plt.plot(overlap_heads_moments_df[stat], marker="o", label=stat)

    # Plot mean and std as band
    plt.plot(x, mean, marker="o", label="Mean", linewidth=2)
    plt.fill_between(x, lower, upper, alpha=0.2, label="±1 Std Dev")

    # Plot layerwise overlap
    plt.plot(
        layerwise_overlap_df.index,
        layerwise_overlap_df["Neighborhood Overlap"],
        marker="o",
        label="Layerwise",
        color="orange",
    )

    plt.xlabel("Layer Index")
    plt.ylabel("Neighborhood Overlap")
    plt.ylim(0, 1)
    plt.title(f"Heads Moments vs Layerwise Overlap ({dataset_name})")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def plot_overlap_text_vs_multimodal(
    layerwise_overlap_text_df,
    layerwise_overlap_multimodal_df,
    heads_mean_overlap_text_np,
    heads_mean_overlap_multimodal_np,
    layerwise_overlap_mmlu_text_df=None,
    figsize=(12, 6),
    title="Overlap Text vs Multimodal",
):
    """
    Plots the overlap between text and multimodal representations.

    Parameters:
    - text_df: pandas DataFrame with text overlap statistics
    - multimodal_df: pandas DataFrame with multimodal overlap statistics
    - figsize: tuple, size of the figure
    - title: plot title
    """

    plt.figure(figsize=figsize)
    plt.plot(
        layerwise_overlap_text_df.index,
        layerwise_overlap_text_df["Neighborhood Overlap"],
        marker="o",
        label="Text Layerwise",
        color="blue",
    )
    plt.plot(
        layerwise_overlap_multimodal_df.index,
        layerwise_overlap_multimodal_df["Neighborhood Overlap"],
        marker="o",
        label="Multimodal Layerwise",
        color="purple",
    )
    plt.plot(
        layerwise_overlap_text_df.index,
        heads_mean_overlap_text_np,
        marker="o",
        label="Text Heads Mean",
        color="red",
    )
    plt.plot(
        layerwise_overlap_multimodal_df.index,
        heads_mean_overlap_multimodal_np,
        marker="o",
        label="Multimodal Heads Mean",
        color="orange",
    )

    if layerwise_overlap_mmlu_text_df is not None:
        plt.plot(
            layerwise_overlap_mmlu_text_df.index,
            layerwise_overlap_mmlu_text_df["Neighborhood Overlap"],
            marker="o",
            label="MMLU Text Layerwise",
            color="green",
        )

    plt.xlabel("Layer Index")
    plt.ylabel("Neighborhood Overlap")
    plt.ylim(0, 1)
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def plot_overlap_layerwise_datasets(
    data_dict,
    figsize=(12, 6),
    title="LLaVA 1.5 7B vs Vicuna 1.5 7B",
):
    """
    Plots the overlap between different datasets.

    Parameters:
    - data_dict: dictionary with dataset names as keys and DataFrames as values
    - figsize: tuple, size of the figure
    - title: plot title
    """

    plt.figure(figsize=figsize)
    for dataset_name, df in data_dict.items():
        plt.plot(
            df.index,
            df["Neighborhood Overlap"],
            marker="o",
            label=dataset_name,
        )

    plt.xlabel("Layer Index")
    plt.ylabel("Neighborhood Overlap")
    plt.ylim(0, 1)
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


## Analysis

In [ ]:
heads_rep_mmlu_path = "../results/neighborhood_overlaps/heads_representations/layer_overlap_llava_hf_llava_1.5_7b_hf_vs_lmsys_vicuna_7b_v1.5_mmlu_test_heads_representations_layer-all_token-last_lm-lmsys-vicuna-7b-v1.5_texts-qa_chat-format_qinst-type-plural_guide-text_continue-fm_seed-42_batch-10maxk-30_downsample-2500.parquet"

heads_rep_scienceqa_text_path = "../results/neighborhood_overlaps/heads_representations/layer_overlap_llava_hf_llava_1.5_7b_hf_vs_lmsys_vicuna_7b_v1.5_ScienceQA_test_heads_representations_layer-all_token-last_lm-lmsys-vicuna-7b-v1.5_texts-qa_chat-format_qinst-type-plural_guide-text_continue-fm_seed-42_batch-20maxk-30.parquet"

heads_rep_scienceqa_images_path = "../results/neighborhood_overlaps/heads_representations/layer_overlap_llava_hf_llava_1.5_7b_hf_vs_lmsys_vicuna_7b_v1.5_ScienceQA_test_heads_representations_layer-all_token-last_lm-lmsys-vicuna-7b-v1.5_images-qa_chat-format_qinst-type-plural_guide-text_continue-fm_seed-42_batch-20maxk-30.parquet"

layers_rep_mmlu_path = "../results/neighborhood_overlaps/layers_representations/layer_overlap_llava_hf_llava_1.5_7b_hf_vs_lmsys_vicuna_7b_v1.5_mmlu_test_layer-all_token-last_texts-qa_chat-format_qinst-type-singular_seed-42_maxk-30_downsample-2500.csv"

layers_rep_scienceqa_text_path = "../results/neighborhood_overlaps/layers_representations/layer_overlap_llava_hf_llava_1.5_7b_hf_vs_lmsys_vicuna_7b_v1.5_ScienceQA_test_layer-all_token-last_lm-lmsys-vicuna-7b-v1.5_texts-qa_chat-format_qinst-type-singular_seed-42maxk-30.csv"

layers_rep_scienceqa_images_path = "../results/neighborhood_overlaps/layers_representations/layer_overlap_llava_hf_llava_1.5_7b_hf_vs_lmsys_vicuna_7b_v1.5_ScienceQA_test_layer-all_token-last_lm-lmsys-vicuna-7b-v1.5_images-qa_chat-format_qinst-type-singular_seed-42maxk-30.csv"

In [ ]:
heads_rep_mmlu_df = pd.read_parquet(heads_rep_mmlu_path)
heads_rep_scienceqa_text_df = pd.read_parquet(heads_rep_scienceqa_text_path)
heads_rep_scienceqa_images_df = pd.read_parquet(heads_rep_scienceqa_images_path)
layers_rep_mmlu_df = pd.read_csv(layers_rep_mmlu_path, index_col=0)
layers_rep_scienceqa_text_df = pd.read_csv(layers_rep_scienceqa_text_path, index_col=0)
layers_rep_scienceqa_images_df = pd.read_csv(
    layers_rep_scienceqa_images_path, index_col=0
)

# Remove prefixes from index and columns
for df in [
    heads_rep_mmlu_df,
    heads_rep_scienceqa_text_df,
    heads_rep_scienceqa_images_df,
]:
    remove_prefixes_str_from_rows_and_columns(df)


heads_rep_scienceqa_text_minus_images_df = (
    heads_rep_scienceqa_text_df - heads_rep_scienceqa_images_df
)

heads_rep_mmlu_np = heads_rep_mmlu_df.to_numpy()
heads_rep_scienceqa_text_np = heads_rep_scienceqa_text_df.to_numpy()
heads_rep_scienceqa_images_np = heads_rep_scienceqa_images_df.to_numpy()
layers_rep_mmlu_np = layers_rep_mmlu_df.to_numpy()
layers_rep_scienceqa_text_np = layers_rep_scienceqa_text_df.to_numpy()
layers_rep_scienceqa_images_np = layers_rep_scienceqa_images_df.to_numpy()

znormalized_heads_rep_scienceqa_text_np = (
    heads_rep_scienceqa_text_np - heads_rep_scienceqa_text_np.mean(axis=1)
) / heads_rep_scienceqa_text_np.std(axis=1)
znormalized_heads_rep_scienceqa_images_np = (
    heads_rep_scienceqa_images_np - heads_rep_scienceqa_images_np.mean(axis=1)
) / heads_rep_scienceqa_images_np.std(axis=1)

znormalized_heads_rep_scienceqa_images_df = (
    heads_rep_scienceqa_images_df - heads_rep_scienceqa_images_df.mean(axis=1)
) / heads_rep_scienceqa_images_df.std(axis=1)
znormalized_heads_rep_scienceqa_text_df = (
    heads_rep_scienceqa_text_df - heads_rep_scienceqa_text_df.mean(axis=1)
) / heads_rep_scienceqa_text_df.std(axis=1)
znormalized_heads_rep_scienceqa_text_minus_images_df = (
    znormalized_heads_rep_scienceqa_text_df - znormalized_heads_rep_scienceqa_images_df
)

mmlu_layers_stats_across_heads = compute_layers_stats_across_heads(heads_rep_mmlu_df)
scienceqa_text_layers_stats_across_heads = compute_layers_stats_across_heads(
    heads_rep_scienceqa_text_df
)
scienceqa_images_layers_stats_across_heads = compute_layers_stats_across_heads(
    heads_rep_scienceqa_images_df
)

layerwise_overlap_data_dict = {
    "MMLU": layers_rep_mmlu_df,
    "ScienceQA Text": layers_rep_scienceqa_text_df,
    "ScienceQA Images": layers_rep_scienceqa_images_df,
}

In [ ]:
upper_limit = 85
lower_limit = 30
step = 5
values = (
    np.arange(lower_limit, upper_limit + 1, step) / 100.0
)  # Convert to [0, 1] range


In [ ]:
# binary_heads_change(
#     heads_rep_scienceqa_text_np,
#     heads_rep_scienceqa_images_np,
# )

In [ ]:
# binary_heads_change(
#     znormalized_heads_rep_scienceqa_text_np,
#     znormalized_heads_rep_scienceqa_images_np,
# )

In [ ]:
upper_limit = 85
lower_limit = 30
step = 5
thresholds = (
    np.arange(lower_limit, upper_limit + 1, step) / 100.0
)  # Convert to [0, 1] range

plot_all_thresholds(
    heads_rep_scienceqa_text_np,
    heads_rep_scienceqa_images_np,
    thresholds,
    "and",
    rows=4,
    cols=3,
)

In [ ]:
plot_all_thresholds(
    znormalized_heads_rep_scienceqa_text_np,
    znormalized_heads_rep_scienceqa_images_np,
    thresholds,
    "xor",
    rows=4,
    cols=3,
)

In [ ]:
plot_overlap_heatmap(
    heads_rep_scienceqa_text_minus_images_df,
    title=r"LLaVa 1.5 7B vs Vicuna 1.5 7B (ScienceQA: $M^{\mathrm{text}}_{l,h} - M^{\mathrm{images}}_{l,h}$)",
)


In [ ]:
plot_overlap_heatmap(
    znormalized_heads_rep_scienceqa_text_minus_images_df,
    title=r"LLaVa 1.5 7B vs Vicuna 1.5 7B (ScienceQA: $Z^{\mathrm{text}}_{l,h} - Z^{\mathrm{images}}_{l,h}$)",
)


In [ ]:
plot_box_plot_layerwise_stats_across_heads(
    heads_rep_mmlu_df,
    dataset_name="MMLU",
)
plot_box_plot_layerwise_stats_across_heads(
    heads_rep_scienceqa_text_df,
    dataset_name="ScienceQA Text",
)
plot_box_plot_layerwise_stats_across_heads(
    heads_rep_scienceqa_images_df,
    dataset_name="ScienceQA Images",
)

In [ ]:
plot_overlap_heads_moments_vs_layerwise(
    mmlu_layers_stats_across_heads,
    layers_rep_mmlu_df,
    dataset_name="MMLU",
    figsize=(10, 6),
)
plot_overlap_heads_moments_vs_layerwise(
    scienceqa_text_layers_stats_across_heads,
    layers_rep_scienceqa_text_df,
    dataset_name="ScienceQA Text",
    figsize=(10, 6),
)
plot_overlap_heads_moments_vs_layerwise(
    scienceqa_images_layers_stats_across_heads,
    layers_rep_scienceqa_images_df,
    dataset_name="ScienceQA Images",
    figsize=(10, 6),
)

In [ ]:
plot_overlap_text_vs_multimodal(
    layers_rep_scienceqa_text_df,
    layers_rep_scienceqa_images_df,
    scienceqa_text_layers_stats_across_heads["mean"].to_numpy(),
    scienceqa_images_layers_stats_across_heads["mean"].to_numpy(),
    figsize=(10, 6),
    title="ScienceQA Text vs Images",
)

In [ ]:
plot_overlap_text_vs_multimodal(
    layers_rep_scienceqa_text_df,
    layers_rep_scienceqa_images_df,
    scienceqa_text_layers_stats_across_heads["mean"].to_numpy(),
    scienceqa_images_layers_stats_across_heads["mean"].to_numpy(),
    layerwise_overlap_mmlu_text_df=layers_rep_mmlu_df,
    figsize=(10, 6),
    title="ScienceQA Text vs Images",
)

In [ ]:
plot_overlap_layerwise_datasets(
    layerwise_overlap_data_dict,
    figsize=(10, 6),
    title="LLaVa 1.5 7B vs Vicuna 1.5 7B",
)

In [ ]:
# plot_layerwise_stats_across_heads(
#     mmlu_layers_stats_across_heads,
#     dataset_name="MMLU",
#     figsize=(10, 6),
# )
# plot_layerwise_stats_across_heads(
#     scienceqa_text_layers_stats_across_heads,
#     dataset_name="ScienceQA Text",
#     figsize=(10, 6),
# )
# plot_layerwise_stats_across_heads(
#     scienceqa_images_layers_stats_across_heads,
#     dataset_name="ScienceQA Images",
#     figsize=(10, 6),
# )

In [ ]:
# plot_overlap_layerwise_vs_heads_mean(
#     layers_rep_mmlu_df,
#     mmlu_layers_stats_across_heads["mean"].to_numpy(),
#     dataset_name="MMLU",
# )
# plot_overlap_layerwise_vs_heads_mean(
#     layers_rep_scienceqa_text_df,
#     scienceqa_text_layers_stats_across_heads["mean"].to_numpy(),
#     dataset_name="ScienceQA Text",
# )
# plot_overlap_layerwise_vs_heads_mean(
#     layers_rep_scienceqa_images_df,
#     scienceqa_images_layers_stats_across_heads["mean"].to_numpy(),
#     dataset_name="ScienceQA Images",
# )